# Options Scanner v4 — Phase 1

**CALL-side scanner with quality scoring gate**

A signal is only generated when conviction score >= 6/10.
If nothing qualifies, the scanner tells you to wait.
Every signal includes exact entry, stop, target, and time stop.

---

| Score | Conviction | Action |
|-------|------------|--------|
| 9-10  | VERY HIGH  | Strong candidate |
| 7-8   | HIGH       | Good candidate |
| 6     | MODERATE   | Minimum threshold — smaller size |
| < 6   | NONE       | No signal — wait |

**Scoring breakdown (10 pts total):**
Regime (2) + RSI (2) + Trend (2) + Volume (2) + Weekly (1) + Support (1)

In [1]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy'])
print('Dependencies ready')

Dependencies ready


In [2]:
# Cell 2 — Load scanner from GitHub
# Replace YOUR_USERNAME with kevin-mumaw
!wget -q https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py

from options_scanner_v4 import (
    run_scan, print_results, deep_dive,
    ALL_SYMBOLS, WATCHLIST, CONFIG
)
print('Scanner loaded')
print(f'Watchlist: {len(ALL_SYMBOLS)} symbols')
print('Groups:', list(WATCHLIST.keys()))

Scanner loaded
Watchlist: 25 symbols
Groups: ['mega_cap_tech', 'financials', 'healthcare', 'consumer', 'etfs', 'industrials', 'energy']


In [3]:
# Cell 3 — Configure
# Edit account size here. Everything else auto-adjusts.

MY_CONFIG = {**CONFIG}  # copy defaults

MY_CONFIG['account_size'] = 2000.00  # update as your account grows

# Optional: scan only specific symbols instead of full watchlist
# MY_SYMBOLS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM']
MY_SYMBOLS = None  # None = scan all 25 symbols

print(f'Account: ${MY_CONFIG["account_size"]:,.0f}')
print(f'Min score to generate signal: {MY_CONFIG["min_score"]}/10')
print(f'Stop loss: -{int(MY_CONFIG["stop_loss_pct"]*100)}% | '
      f'Profit target: +{int(MY_CONFIG["profit_target_pct"]*100)}% | '
      f'Time stop: {MY_CONFIG["time_stop_dte"]} DTE')

Account: $2,000
Min score to generate signal: 6/10
Stop loss: -35% | Profit target: +75% | Time stop: 21 DTE


In [7]:
# Cell 4 — Run the scan
# This is the main cell. Run this daily.
results = run_scan(symbols=MY_SYMBOLS, config=MY_CONFIG)
print_results(results)


  Fetching market regime (QQQ)...
  Regime: BULLISH — QQQ 717.54 | MA50 642.01 (+11.8%) | MA200 613.6 (+16.9%)

  Scanning 25 symbols......................... done.


██████████████████████████████████████████████████████████████
  OPTIONS SCANNER — Phase 1  |  2026-05-24 00:59
  Account: $2,000  |  Scanned: 25 symbols
██████████████████████████████████████████████████████████████

  MARKET REGIME: 🟢 BULLISH
  QQQ 717.54 | MA50 642.01 (+11.8%) | MA200 613.6 (+16.9%)

  3 signal(s) found:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SIGNAL #1  |  COST — BUY CALL  |  Score: 8/10  [HIGH]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  WHY THIS TRADE:
    ✓ Regime [2/2] BULLISH — QQQ above both MAs
    ~ RSI     [1/2] Neutral — RSI 54 (room to run)
    ✓ Trend   [2/2] Clean uptrend — price > MA20 > MA50
    ✓ Weekly  [1/1] Weekly uptrend confirmed
    ~ Volume  [1/2] Neutral volume — no clear institutional signal
    ✓ Support [1/1] Near MA50 support

In [5]:
# Cell 5 — Deep dive on a specific symbol
# Use this to investigate any symbol in detail,
# whether it appeared in the scan or not.

SYMBOL = 'GOOGL'  # change this

deep_dive(SYMBOL, config=MY_CONFIG)


  Fetching regime and analyzing GOOGL...

██████████████████████████████████████████████████████████████
  OPTIONS SCANNER — Phase 1  |  2026-05-24 00:56
  Account: $2,000  |  Scanned: 1 symbols
██████████████████████████████████████████████████████████████

  MARKET REGIME: 🟢 BULLISH
  QQQ 717.54 | MA50 642.01 (+11.8%) | MA200 613.6 (+16.9%)

  1 signal(s) found:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SIGNAL #1  |  GOOGL — BUY CALL  |  Score: 6/10  [MODERATE]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  WHY THIS TRADE:
    ✓ Regime [2/2] BULLISH — QQQ above both MAs
    ~ RSI     [1/2] Neutral — RSI 50 (room to run)
    ~ Trend   [1/2] Mixed but above MA50 support
    ✓ Weekly  [1/1] Weekly uptrend confirmed
    ~ Volume  [1/2] Neutral volume — no clear institutional signal
    ✗ Support [0/1] Extended +12.3% above MA50 — wait for pullback

  STOCK:
    Price: $382.97   RSI: 50   Trend: MIXED
    MA20:  $385.48 (-0.7%)   MA50: $341.14 (

In [6]:
# Cell 6 — Review all scores from last scan
# See every symbol's score ranked highest to lowest.

import pandas as pd

rows = []
for r in results['all_results']:
    if not r.get('error'):
        rows.append({
            'Symbol':    r['symbol'],
            'Score':     r['score'],
            'Conviction':r['conviction'],
            'Trend':     r.get('trend','—'),
            'RSI':       r.get('rsi','—'),
            'Volume':    r.get('vol_data',{}).get('label','—'),
            'Weekly':    r.get('weekly',{}).get('trend','—'),
            'Has Option':r.get('option') is not None,
        })

df = pd.DataFrame(rows).sort_values('Score', ascending=False)
print(df.to_string(index=False))

Symbol  Score Conviction     Trend  RSI            Volume    Weekly  Has Option
  COST      8       HIGH   UPTREND 53.6           NEUTRAL   UPTREND        True
   JPM      7       HIGH   UPTREND 48.8           NEUTRAL     MIXED        True
     V      7       HIGH   UPTREND 52.9           NEUTRAL     MIXED        True
   BAC      7       HIGH   UPTREND 47.5 MILD_ACCUMULATION     MIXED        True
   XLF      7       HIGH   UPTREND 54.0           NEUTRAL     MIXED        True
   XOM      7       HIGH     MIXED 53.0           NEUTRAL   UPTREND        True
  AAPL      6   MODERATE   UPTREND 91.1           NEUTRAL   UPTREND        True
  MSFT      6   MODERATE   UPTREND 55.1           NEUTRAL     MIXED        True
 GOOGL      6   MODERATE     MIXED 49.8           NEUTRAL   UPTREND        True
  AMZN      6   MODERATE     MIXED 43.3           NEUTRAL   UPTREND        True
  NVDA      6   MODERATE   UPTREND 62.5           NEUTRAL   UPTREND        True
    GS      6   MODERATE   UPTREND 73.7 